In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 7.1 Complex Analysis I: Analytic Functions and the Residue Theorem

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VII — Quantum Statistical Mechanics",
    number="7.1",
    title="Complex Analysis I: Analytic Functions and the Residue Theorem",
    blurb="A volume on quantum statistics opens with a detour that is not one. The functions "
    "of a complex variable that possess a derivative are rigid objects: their values "
    "on a curve fix them everywhere inside, their real and imaginary parts solve "
    "Laplace's equation, and their closed-contour integrals are governed entirely by "
    "a handful of residues at their singularities. That rigidity turns impossible "
    "integrals into two-line calculations — including, soon, the integrals that carry "
    "all of quantum statistics.",
    difficulty="advanced",
    estimate="185–225 min",
)

## Notebook overview

Volume VII begins the way Volumes 0, V, and VI did: with its mathematics, built once and properly,
before any physics is allowed to depend on it. The mathematics this volume runs on is the analysis
of functions of a **complex variable**. The claim deserves justification, because at first sight
quantum statistical mechanics is about counting states and weighting them by $e^{-\beta E}$ — where
is the complex plane in that? Everywhere, it turns out. The Bose and Fermi integrals that carry the
photon gas, the degenerate electron gas, and Bose–Einstein condensation are integrals wrapped
around branch cuts; the frequency sums of thermal physics (the Matsubara sums of
[§7.2](complex-analysis-applications.ipynb)) are evaluated
by trading a sum for the poles of a function that has one at every term; the Stirling approximation
that Volume V leaned on is the shallowest case of an asymptotic method (steepest descent) that
lives entirely in the complex plane; and the response functions of matter obey exact relations —
the Kramers–Kronig relations of [§7.2](complex-analysis-applications.ipynb) — that are theorems
about analyticity, with causality as the
only physical input. A subject whose integrals, sums, asymptotics, and response theory all route
through the complex plane deserves to have that plane built carefully.

So this notebook builds the core machinery: what it means for a complex function to be
**differentiable** (the Cauchy–Riemann equations, and the rigidity they impose), the
**multivaluedness** of the logarithm and the branch cuts that tame it, **contour integration** and
Cauchy's theorem, the **integral formula** that lets a boundary dictate an interior, **Laurent
series** and the classification of singularities, and the **residue theorem** — the machine that
converts closed-contour integrals into arithmetic. The chapter's climax spends that machinery on
the four classic families of real integrals, each chosen with intent: the Fourier-type family is
the Lorentzian-lineshape and exponential-decay pair behind
[§6.24](../06-quantum-mechanics/time-dependent-perturbation.ipynb), and the closing keyhole integral
is one Boltzmann factor away from the Bose and Fermi integrals of [§7.3](statistical-toolkit.ipynb).

A word on curriculum, recorded deliberately. This course ordinarily teaches a complete *minimal*
toolkit — each technique introduced exactly when needed, to exactly the depth needed. The
three-notebook arsenal that opens this volume is a deliberate exception: complex analysis is
developed in full, Cauchy–Riemann equations and all, because the subject is too central — to
physics and to the education of anyone who computes for a living — to be issued on a need-to-know
basis. It is a detour that is not one.

> **Conventions (fixed for the whole volume).** We use the **principal branch** of the logarithm
> and the argument, $\arg z \in (-\pi, \pi]$, which is exactly the convention of `numpy.log` and
> `numpy.angle`; its branch cut lies along the negative real axis. Closed contours are traversed
> **counterclockwise** (positive orientation) unless stated otherwise. Contour integrals are
> computed numerically by parametrization, $\oint f\,dz = \int f(z(t))\,z'(t)\,dt$, with
> `numpy.trapezoid` on $N \gtrsim 10^4$ points — and for a *closed* contour this humble rule is
> secretly spectral: on a smooth periodic integrand its error falls geometrically with $N$, which
> is why the checks below routinely reach $10^{-15}$.
>
> **How to read the checks.** Each exercise closes with a `validate` call against an independent
> fact: Cauchy–Riemann residuals at rounding level for $z^2$ and $e^z$ against an exactly-2
> violation for $\bar z$; the $2\pi i$ jump of the logarithm across its cut; $\oint z^2\,dz$
> vanishing to machine precision while $\oint \bar z\,dz$ measures the enclosed area exactly; the
> integral formula reconstructing $e^a$ from boundary values; residues extracted by small-circle
> contours agreeing with the limit formulas; and all four real-integral families matching
> `scipy.integrate.quad` — except where quadrature itself strains, which is part of the lesson.
> A ✓ is strong evidence; a ✗ is a prompt to *locate the discrepancy*.
>
> **Scope.** One complex variable, computationally, from analyticity to the residue theorem and
> its classic applications. The physicist's payload — principal values and Sokhotski–Plemelj, the
> Kramers–Kronig relations, Matsubara sums, steepest descent — is
> [§7.2](complex-analysis-applications.ipynb); the statistical toolkit
> (Gamma and zeta functions, polylogarithms, the Bose and Fermi integrals, the Sommerfeld
> expansion) is [§7.3](statistical-toolkit.ipynb). Conformal mapping is named as a horizon, not
> developed. See Arfken, Weber &
> Harris, *Mathematical Methods for Physicists*; Ablowitz & Fokas, *Complex Variables*; and — for
> the geometrically curious — Needham, *Visual Complex Analysis*. Cross-reference
> [§6.1](../06-quantum-mechanics/complex-vector-spaces.ipynb) (the algebra
> of $\mathbb{C}$), Volume III (two-dimensional electrostatics and harmonic potentials),
> [§5.3](../05-classical-stat-mech/large-n-limit.ipynb)
> (Stirling, re-derived by steepest descent in [§7.2](complex-analysis-applications.ipynb)), and
> [§6.24](../06-quantum-mechanics/time-dependent-perturbation.ipynb) (the Lorentzian lineshape).

## Theory in brief

### Analyticity and the Cauchy–Riemann equations

Write $f(z) = u(x,y) + i\,v(x,y)$ for $z = x + iy$. The function is **analytic** (holomorphic) at
$z$ if the complex derivative

$$
f'(z) = \lim_{\Delta z \to 0} \frac{f(z + \Delta z) - f(z)}{\Delta z}
$$

exists — with the *same* value for every direction of approach. That innocuous clause is the whole
subject. Approach horizontally ($\Delta z = h$, real) and the limit is $\partial f/\partial x
= u_x + i v_x$. Approach vertically ($\Delta z = ih$) and it is $\tfrac{1}{i}\,\partial f/\partial
y = v_y - i u_y$. Demanding these agree, real and imaginary parts separately, gives the
**Cauchy–Riemann equations**,

```{math}
:label: eq-cauchy-riemann
\frac{\partial u}{\partial x} = \frac{\partial v}{\partial y},
\qquad
\frac{\partial u}{\partial y} = -\frac{\partial v}{\partial x}.
```

Two real constraints on four partial derivatives: complex differentiability is not a notation but
a *condition*, and most smooth functions of $(x, y)$ fail it. It is also sharply checkable:
$z^2$ and $e^z$ satisfy {eq}`eq-cauchy-riemann` (numerically, residuals near $10^{-10}$), while
$\bar z$ and $|z|^2$ violate it at order one — for $\bar z$ the violation is *exactly* $2$.
The analytic functions are the aristocracy of functions, and admission is strict.

### Harmonic consequences

Differentiate {eq}`eq-cauchy-riemann` once more and cross the equations:
$u_{xx} = v_{yx} = v_{xy} = -u_{yy}$. The real part of an analytic function — and, by the same
argument, the imaginary part — satisfies **Laplace's equation**,

```{math}
:label: eq-harmonic
\nabla^2 u = \frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2} = 0,
\qquad
\nabla^2 v = 0 .
```

Every analytic function therefore packages *two* solutions of the two-dimensional Laplace equation
— the equation of the charge-free electrostatics of Volume III — and the Cauchy–Riemann equations
make their level curves mutually orthogonal: equipotentials and field lines, delivered as a pair.
This is the doorway to **conformal mapping**, the classical art of solving two-dimensional
electrostatics and ideal flow by bending the plane analytically; we name the horizon and move on.

### Elementary functions and branch cuts

The exponential $e^z = e^x(\cos y + i \sin y)$ is **entire** — analytic everywhere. Its inverse
cannot be: since $e^{z + 2\pi i} = e^z$, the logarithm is **multivalued**,

```{math}
:label: eq-branch-cuts
\log z = \ln|z| + i \arg z,
\qquad
\arg z \in (-\pi, \pi] \ \text{(principal branch, the convention of numpy.log)},
```

and choosing the principal branch introduces a **branch cut** along the negative real axis, across
which $\log z$ jumps by $2\pi i$. Fractional powers inherit the cut through $z^{\alpha} =
e^{\alpha \log z}$: follow $\sqrt z$ continuously once around the origin and it returns to *minus*
its starting value — the function lives on two sheets, and the cut is where we have chosen to
staple them. Branch cuts are not pathology but bookkeeping, and the bookkeeping becomes
load-bearing twice below: in the keyhole integral of Exercise 8, where the cut is the mechanism
that makes the method work, and in the polylogarithms of [§7.3](statistical-toolkit.ipynb), which
carry their physics on their cuts.

### Contour integrals and Cauchy's theorem

The integral of $f$ along a path is defined by parametrization,
$\int_C f\,dz = \int_{t_0}^{t_1} f(z(t))\,z'(t)\,dt$, and this is exactly how we compute it
throughout (`numpy.trapezoid` on a discretized contour). The first pillar of the subject is
**Cauchy's theorem**: for $f$ analytic on and inside a closed contour $C$,

```{math}
:label: eq-cauchy-theorem
\oint_C f(z)\,dz = 0 .
```

(Verified below to $\sim 10^{-16}$.) The classical proof runs through Green's theorem and
the Cauchy–Riemann equations {eq}`eq-cauchy-riemann`; Ablowitz & Fokas, *Complex Variables*,
Ch. 2, carries it out in full. The practical reading is that contours may be
**deformed** freely through analytic territory without changing the integral — the single
most-used move in the subject, and the one that powers everything after. The converse spectacle is
just as instructive: for the non-analytic $\bar z$, the closed integral does not merely fail to
vanish, it *measures geometry*, $\oint \bar z\,dz = 2i \times (\text{enclosed area})$, exactly.

### The Cauchy integral formula

For $f$ analytic on and inside $C$ and $a$ any interior point,

```{math}
:label: eq-integral-formula
f(a) = \frac{1}{2\pi i} \oint_C \frac{f(z)}{z - a}\,dz .
```

The derivation is one contour deformation: shrink $C$ to a small circle about $a$, where
$f(z) \approx f(a)$ and the integral of $(z-a)^{-1}$ supplies the $2\pi i$; Arfken, Weber &
Harris, Ch. 11, writes it out line by line.
This is the rigidity thesis made literal: the values of $f$ on the *boundary* determine its value
at every *interior* point — knowing an analytic function on a curve is knowing it inside.
Differentiating under the integral gives $f^{(n)}(a)$ for every $n$: an analytic function is
automatically infinitely differentiable, and its Taylor series converges to it. (Compare real
analysis, where differentiability once buys nothing further; here it buys everything, unasked.)

### Laurent series, poles, and residues

Around an isolated singularity at $a$, an analytic function expands in a **Laurent series**, a
Taylor series extended to negative powers (the expansion follows from the integral formula
applied to an annulus; Arfken, Weber & Harris, Ch. 11, derives it in full),

```{math}
:label: eq-laurent
f(z) = \sum_{n=-\infty}^{\infty} c_n (z - a)^n,
\qquad
\oint (z-a)^n\,dz = 2\pi i\,\delta_{n,-1},
\qquad
\operatorname{Res}_{z=a} f = c_{-1} .
```

If the negative powers terminate at $n = -m$ the singularity is a **pole of order $m$**; if they
do not terminate it is an **essential singularity** ($e^{1/z}$ is the standard specimen). The
orthogonality relation in the middle of {eq}`eq-laurent` — direct parametrization shows every
power of $(z-a)$ integrates to zero around a circle *except* $n = -1$ — is why the coefficient
$c_{-1}$, the **residue**, is the sole survivor of closed-contour integration, and hence the only
number the theorem below needs. Extracting residues is a craft with three tools: the limit formula
$c_{-1} = \lim_{z\to a}(z-a)f(z)$ for simple poles, a derivative formula for higher order, and —
always available, whatever the order — the *numerical small-circle contour*, which reads
$c_{-1}$ off directly.

### The residue theorem

Combine Cauchy's theorem (deform the contour into small circles around each enclosed singularity)
with the survival of $c_{-1}$ (evaluate each small circle), and the pillars merge into the machine:

```{math}
:label: eq-residue-theorem
\oint_C f(z)\,dz = 2\pi i \sum_{\text{poles } a_k \text{ inside } C} \operatorname{Res}_{z=a_k} f .
```

Three moves, endlessly repeated: **close** a contour, **locate** the poles, **read off** the
residues. Integrals that resist every elementary technique fall in two lines.

### The four classic families of real integrals

The machine's classic applications, each verified against quadrature below:

```{math}
:label: eq-real-integrals
\begin{aligned}
\text{(i) rational:}\quad & \int_{-\infty}^{\infty} \frac{dx}{1+x^4} = \frac{\pi}{\sqrt 2}
&& \text{(close a semicircle above)} \\[2pt]
\text{(ii) trigonometric:}\quad & \int_0^{2\pi} \frac{d\theta}{2+\cos\theta} = \frac{2\pi}{\sqrt 3}
&& (z = e^{i\theta} \text{ walks the unit circle}) \\[2pt]
\text{(iii) Fourier:}\quad & \int_{-\infty}^{\infty} \frac{e^{ikx}}{x^2+a^2}\,dx
  = \frac{\pi}{a}\,e^{-|k|a}
&& \text{(Jordan's lemma picks the half-plane)} \\[2pt]
\text{(iv) keyhole:}\quad & \int_0^{\infty} \frac{x^{\alpha-1}}{1+x}\,dx
  = \frac{\pi}{\sin \pi\alpha}, \quad 0<\alpha<1
&& \text{(a contour wrapped around a branch cut).}
\end{aligned}
```

For (i), the closing semicircle contributes nothing because the integrand decays faster than
$1/|z|$; for (iii), **Jordan's lemma** closes $e^{ikz}$ in the half-plane where it *decays*
($\operatorname{Im} z > 0$ for $k > 0$), and the result is the **Lorentzian–exponential pair**
behind every linewidth and lifetime in [§6.24](../06-quantum-mechanics/time-dependent-perturbation.ipynb)
— a pole's imaginary part *is* a decay rate. Family
(iii) is also where the contour method beats the computer outright: `scipy.integrate.quad` warns
and drifts on the oscillatory integrand while the residue answer is exact. And family (iv), the
**keyhole**, is a deliberate pre-echo: replace $1 + x$ by $e^x - 1$ or $e^x + 1$ and these are the
Bose and Fermi integrals of [§7.3](statistical-toolkit.ipynb) — same anatomy, same cut, same three
moves.

## Setup

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import IntegrationWarning, quad

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Conventions: the PRINCIPAL branch of log and arg
# (numpy.log / numpy.angle, arg ∈ (−π, π], cut on the negative real axis) unless an
# exercise explicitly builds another branch; closed contours run COUNTERCLOCKWISE;
# contour integrals are computed by parametrization with numpy.trapezoid, N ≳ 10^4.
# On a CLOSED contour the trapezoid rule is spectrally accurate (the integrand is
# periodic in t), so these checks reach near machine precision at modest N — until
# a pole crowds the path, which Exercise 4 measures deliberately.


def contour_integral(f, z_of_t, dz_of_t, t0, t1, N=20001):
    """The path integral ∫ f(z) dz along z(t), t from t0 to t1, by numpy.trapezoid.

    The workhorse of the notebook: discretize t, evaluate f(z(t))·z'(t), and hand the
    samples to numpy.trapezoid. For a CLOSED contour the integrand is periodic in t and
    the trapezoid rule converges geometrically with N (its error is controlled by the
    distance of the nearest singularity of f from the path), which is why closed-contour
    checks below sit at 1e-15 with N = 20001. Passing t0 > t1 reverses the orientation,
    which flips the sign — used for the clockwise inner circle of the keyhole contour.

    Parameters
    ----------
    f : callable
        The integrand f(z); must accept complex numpy arrays.
    z_of_t, dz_of_t : callable
        The path z(t) and its derivative dz/dt.
    t0, t1 : float
        Parameter limits; t0 > t1 gives the reversed (negative) orientation.
    N : int, optional
        Number of sample points (default 20001).

    Returns
    -------
    complex
        The contour integral ∫ f(z(t)) z'(t) dt.
    """
    t = np.linspace(t0, t1, N)
    return np.trapezoid(f(z_of_t(t)) * dz_of_t(t), t)


def circle(c, R):
    """Parametrization pair (z(t), dz/dt) for the counterclockwise circle z = c + R e^(it).

    A tiny factory, but it keeps every call site honest about the two things that are
    silently easy to get wrong: the orientation (counterclockwise, t from 0 to 2π, the
    positive orientation of the residue theorem) and the Jacobian dz/dt = iR e^(it)
    that the parametrized integral must carry.

    Parameters
    ----------
    c : complex
        Centre of the circle.
    R : float
        Radius.

    Returns
    -------
    tuple of callable
        (z_of_t, dz_of_t), ready for :func:`contour_integral`.
    """
    return (lambda t: c + R * np.exp(1j * t), lambda t: 1j * R * np.exp(1j * t))


def cauchy_riemann_residual(f, z0, h=1e-5):
    """Max Cauchy–Riemann violation max(|u_x − v_y|, |u_y + v_x|) at z0, by central differences.

    Writes f = u + iv and forms the four partials from two symmetric differences:
    (f(z0+h) − f(z0−h))/2h gives u_x + i·v_x, and (f(z0+ih) − f(z0−ih))/2h gives
    u_y + i·v_y (eq-cauchy-riemann). The step h = 1e-5 balances the O(h^2) truncation
    error against the O(eps/h) rounding error, putting the floor near 1e-10: an analytic
    function lands on that floor, a non-analytic one shows an O(1) violation — there is
    no middle ground, which is the point of Exercise 1.

    Parameters
    ----------
    f : callable
        The function to test; must accept complex scalars.
    z0 : complex
        The point at which to test analyticity.
    h : float, optional
        Central-difference step (default 1e-5).

    Returns
    -------
    float
        max(|u_x − v_y|, |u_y + v_x|); ~1e-10 for analytic f, O(1) otherwise.
    """
    fx = (f(z0 + h) - f(z0 - h)) / (2 * h)  # ∂f/∂x = u_x + i v_x
    fy = (f(z0 + 1j * h) - f(z0 - 1j * h)) / (2 * h)  # ∂f/∂y = u_y + i v_y
    ux, vx = fx.real, fx.imag
    uy, vy = fy.real, fy.imag
    return max(abs(ux - vy), abs(uy + vx))


def residue_numeric(f, a, r=0.2, N=20001):
    """The residue of f at a, read off as (1/2πi) ∮ f dz on the small circle |z − a| = r.

    Only the 1/(z − a) term of the Laurent series survives integration around the pole
    (eq-laurent), so one small-circle contour integral IS the residue — no limit formula,
    no derivatives, and the pole's order is irrelevant. Choose r small enough that no
    OTHER singularity is enclosed (the extraction is only as honest as that geometry),
    but not so small that the integrand's growth near the pole, ~ r^(−m) for an order-m
    pole, erodes the quadrature accuracy.

    Parameters
    ----------
    f : callable
        The function whose residue is wanted; accepts complex arrays.
    a : complex
        Location of the isolated singularity.
    r : float, optional
        Radius of the extraction circle (default 0.2).
    N : int, optional
        Sample points for the contour (default 20001).

    Returns
    -------
    complex
        The residue of f at a.
    """
    z_of_t, dz_of_t = circle(a, r)
    return contour_integral(f, z_of_t, dz_of_t, 0.0, 2 * np.pi, N) / (2j * np.pi)


def close_and_sum_residues(residues):
    """The residue theorem's right-hand side, 2πi Σ residues (eq-residue-theorem).

    Deliberately trivial arithmetic, packaged: closing a contour and summing residues is
    the three-move machine of this notebook (close, locate, read off), and the factor
    2πi is the single most-forgotten constant in the subject. The analysis remains the
    caller's job — that the closing arc vanishes (the decay condition of Exercise 6,
    Jordan's lemma in Exercise 7) and that exactly the enclosed poles are listed.

    Parameters
    ----------
    residues : sequence of complex
        The residues at the poles enclosed by the (counterclockwise) closed contour.

    Returns
    -------
    complex
        2πi times the sum of the residues.
    """
    return 2j * np.pi * np.sum(np.asarray(residues, dtype=complex))

## Exercise 1 — Analyticity is a sharp property

Complex differentiability is not a formality: it either holds, with far-reaching consequences, or
it fails completely, and a one-line numerical test tells us which. We make the test and run it.
Cite {eq}`eq-cauchy-riemann`, {eq}`eq-harmonic`.

1. Derive the Cauchy–Riemann equations by equating the horizontal and vertical limits of the
   difference quotient defining $f'(z)$.
2. Using the `cauchy_riemann_residual` helper (central differences of step $h = 10^{-5}$),
   evaluate the violation $\max(|u_x - v_y|,\ |u_y + v_x|)$ for $z^2$, $e^z$, $\bar z$, and
   $|z|^2$ at several points of the plane.
3. Confirm that the analytic pair sits at rounding level ($\sim 10^{-9}$ or below) while $\bar z$
   and $|z|^2$ violate at order one — and show *why* the $\bar z$ violation is exactly $2$.
4. Verify with a five-point Laplacian (central differences via plain `numpy` arithmetic) that
   $\operatorname{Re} z^2 = x^2 - y^2$ is harmonic, and state the consequence: the real and
   imaginary parts of analytic functions are two-dimensional electrostatic potentials.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    cr_analytic_max < 1e-8,
    "the analytic pair z^2, e^z satisfies the Cauchy–Riemann equations to rounding level",
    f"worst residual {cr_analytic_max:.1e}",
)
validate.close(
    cr_zbar,
    2.0,
    "the Cauchy–Riemann violation of conj(z) is exactly 2 (u_x − v_y = 1 − (−1))",
    rtol=1e-12,
)
validate.check(
    cr_mod > 0.5 and abs(lap_re) < 1e-6,
    "|z|^2 violates Cauchy–Riemann at O(1), while Re z^2 is harmonic (∇²u ≈ 0)",
    f"violation {cr_mod:.2f}, Laplacian {lap_re:.1e}",
)

## Exercise 2 — The branch cut of the logarithm

The logarithm's multivaluedness is bookkeeping we must own before fractional powers, the keyhole
contour of Exercise 8, and the polylogarithms of [§7.3](statistical-toolkit.ipynb). We make the
cut visible and watch a function
change sheets. Cite {eq}`eq-branch-cuts`.

1. Evaluate `numpy.log` just above and just below the negative real axis (offsets $\pm 10^{-9}i$
   at $z = -1$ — never *on* the axis, where the convention alone decides) and exhibit the $2\pi i$
   jump across the cut.
2. Plot $\operatorname{Im}\log z = \arg z$ over the plane with `numpy.angle` on a grid, using a
   *cyclic* colormap so the only visible seam is the genuine cut on the negative real axis.
3. Show that $\sqrt z = e^{\frac{1}{2}\log z}$ inherits the cut: track $\sqrt z$ *continuously*
   along the unit circle (unwrap the phase with `numpy.unwrap` + `numpy.angle`) and watch it
   return to *minus* its starting value after one full loop.
4. State the convention adopted for the rest of the volume (principal branch, cut on the negative
   real axis) and why a choice must be made at all.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    log_jump,
    2j * np.pi,
    "the principal logarithm jumps by 2πi across its branch cut",
    rtol=1e-6,
)
validate.close(
    sqrt_cont[-1],
    -sqrt_cont[0],
    "√z, continued around the origin, returns to minus itself (two sheets)",
    rtol=1e-9,
)

## Exercise 3 — Cauchy's theorem, and an integral that measures area

Closed-contour integrals of analytic functions vanish — and can therefore be deformed at will —
while the failure for a non-analytic integrand is so total that the integral turns into a
surveyor's tool. Cite {eq}`eq-cauchy-theorem`.

1. Compute $\oint z^2\,dz$ around the unit circle with the `contour_integral` helper
   (parametrization $z(t) = e^{it}$, integrated with `numpy.trapezoid`).
2. Confirm the result vanishes to $\sim 10^{-15}$, and check the deformation property by
   repeating on an ellipse $z(t) = a\cos t + i\,b\sin t$: still zero.
3. Compute $\oint \bar z\,dz$ around both contours and confirm the value is $2i \times
   (\text{enclosed area})$ in both cases — $2\pi i$ for the unit disk, $2i\,\pi a b$ for the
   ellipse.
4. Interpret: analytic integrands forget the path; the non-analytic $\bar z$ remembers
   everything, down to the geometry.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    abs(int_z2_circle) < 1e-12 and abs(int_z2_ellipse) < 1e-12,
    "Cauchy's theorem: ∮ z^2 dz = 0 on the circle and on the deformed (elliptical) contour",
    f"|circle| = {abs(int_z2_circle):.1e}, |ellipse| = {abs(int_z2_ellipse):.1e}",
)
validate.close(
    int_zbar_circle,
    2j * np.pi,
    "∮ conj(z) dz = 2i × Area detects the geometry: 2πi for the unit disk",
    rtol=1e-9,
)
validate.close(
    int_zbar_ellipse,
    2j * area_ellipse,
    "∮ conj(z) dz = 2i × Area holds on the ellipse too (2i·πab)",
    rtol=1e-9,
)

## Exercise 4 — The Cauchy integral formula: the boundary knows the interior

Rigidity made computational: the values of an analytic function on a closed curve determine its
value at every point inside, and we can carry out the reconstruction numerically.
Cite {eq}`eq-integral-formula`.

1. For $f = e^z$ and an interior point $a$, evaluate $\frac{1}{2\pi i}\oint f(z)/(z-a)\,dz$ on
   the unit circle with `contour_integral`.
2. Confirm it reproduces $e^a$ to at least six digits, and repeat for a second function
   ($f = \cos z$, `numpy.cos`) and several points $a$.
3. Move $a$ toward the contour ($|a| = 0.5,\ 0.9,\ 0.99,\ 0.999$ at fixed $N$) and watch the
   accuracy degrade — explain why (the pole of the integrand approaches the discretized path,
   and the trapezoid rule's geometric convergence rate collapses).
4. State the consequences in prose: analytic functions are infinitely differentiable and equal
   their Taylor series; knowing $f$ on a curve is knowing $f$ inside.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    rebuilt,
    np.exp(a1),
    "the Cauchy integral formula reconstructs interior values from the boundary (f = e^z)",
    rtol=1e-10,
)
validate.check(
    err_cos_max < 1e-9,
    "the reconstruction is universal: f = cos z rebuilt at several interior points",
    f"worst error {err_cos_max:.1e}",
)
validate.check(
    err_near[0.5] < 1e-12 and err_near[0.999] > 1e3 * max(err_near[0.5], 1e-15),
    "accuracy degrades as the pole approaches the discretized contour (geometric rate e^(−N·d))",
    f"error {err_near[0.5]:.1e} at |a|=0.5 vs {err_near[0.999]:.1e} at |a|=0.999",
)

## Exercise 5 — Laurent series and residues

Around an isolated singularity a function expands in powers both positive and negative, and
exactly one coefficient survives closed-contour integration. We verify the survival, then extract
residues three ways. Cite {eq}`eq-laurent`.

1. Show by direct parametrization (`contour_integral` on a circle around $a$) that
   $\oint (z-a)^n\,dz = 2\pi i\,\delta_{n,-1}$ for $n = -3, \dots, 2$.
2. For $f(z) = 1/\big((z-\tfrac12)(z+2)\big)$, compute the residue at $z = \tfrac12$ by the
   simple-pole limit formula and by `residue_numeric` (small-circle extraction); confirm both
   give $0.4$.
3. For $f(z) = e^z/z^2$, write the Laurent expansion around $0$, identify the residue (the $1/z$
   coefficient, $= 1$), confirm with `residue_numeric`, and read off the first four Laurent
   coefficients numerically via $c_n = \frac{1}{2\pi i}\oint f(z)/z^{\,n+1}\,dz$.
4. Classify in prose: pole orders, essential singularities ($e^{1/z}$), and why the residue is
   the only survivor.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    [survival[-1], survival[-2], survival[1]],
    [2j * np.pi, 0.0, 0.0],
    "∮ (z − a)^n dz = 2πi δ_(n,−1): only the residue term survives integration",
    atol=1e-10,
)
validate.close(
    [res_limit, res_small_circle, res_double],
    [0.4, 0.4, 1.0],
    "residues by the limit formula and by small-circle contour extraction agree",
    rtol=1e-6,
)
validate.close(
    laurent_num,
    laurent_exact,
    "the small-circle contour reads off the whole Laurent series of e^z/z^2 (c_n = 1/(n+2)!)",
    rtol=1e-8,
)

## Exercise 6 — The residue theorem, and the rational family

The machine assembled and first used: closed contours reduce to residue arithmetic, and the first
family of impossible-looking real integrals falls. Cite {eq}`eq-residue-theorem`,
{eq}`eq-real-integrals`.

1. Verify the residue theorem numerically with `contour_integral`: for the two-pole
   $f(z) = 1/\big((z-\tfrac12)(z+2)\big)$ of Exercise 5, integrate on a contour enclosing *both*
   poles and confirm $\oint f\,dz = 2\pi i \sum \text{Res} = 0$ (the residues cancel), then on a
   contour enclosing only $z = \tfrac12$ and confirm the single-residue value.
2. Evaluate $\int_{-\infty}^{\infty} dx/(1+x^4)$ by closing a semicircle in the upper half-plane:
   locate the two enclosed poles (the fourth roots of $-1$), extract their residues
   with `residue_numeric`, check them against the analytic values $-p/4$, and assemble the answer
   $\pi/\sqrt 2$ with `close_and_sum_residues`.
3. Justify discarding the arc: integrate the semicircular arc numerically at $R = 10, 30, 100$
   with `contour_integral`, watch it shrink like $R^{-3}$, and state the general decay condition.
4. Confirm against `scipy.integrate.quad`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    loop_both,
    0.0,
    "the residue theorem with cancelling residues: ∮ f dz = 2πi(0.4 − 0.4) = 0",
    atol=1e-10,
)
validate.close(
    loop_one,
    2j * np.pi * 0.4,
    "a contour enclosing only z = 1/2 picks up 2πi times that residue alone",
    rtol=1e-9,
)
validate.close(
    res_extracted,
    res_analytic,
    "small-circle extraction matches the analytic residues −p/4 at both enclosed poles",
    rtol=1e-9,
)
validate.close(
    I_residue,
    np.pi / np.sqrt(2),
    "∫ dx/(1 + x^4) = π/√2 by the residue theorem",
    rtol=1e-9,
)
validate.close(
    I_residue,
    I_quad,
    "the residue answer matches scipy.integrate.quad",
    rtol=1e-8,
)

## Exercise 7 — Trigonometric and Fourier families: where residues beat the computer

Two more closures — and the demonstration that the contour method wins computationally exactly
where brute-force quadrature strains: on oscillatory integrands. Cite {eq}`eq-real-integrals`.

1. Convert $\int_0^{2\pi} d\theta/(2+\cos\theta)$ to a unit-circle contour integral via
   $z = e^{i\theta}$ (so $\cos\theta = (z+1/z)/2$ and $d\theta = dz/(iz)$), evaluate it by
   residues (`residue_numeric` + `close_and_sum_residues`, cross-checked by `contour_integral`
   on the unit circle), and confirm $2\pi/\sqrt 3$ against `scipy.integrate.quad` in $\theta$.
2. Evaluate $\int_{-\infty}^{\infty} e^{ikx}/(x^2+a^2)\,dx$ for $k = a = 1$ by closing in the
   half-plane where $e^{ikz}$ decays (Jordan's lemma), obtaining $(\pi/a)e^{-|k|a}$; the real
   part is the cosine integral.
3. Benchmark against `scipy.integrate.quad` with a large `limit`, recording its warning and its
   $\sim 5\times10^{-5}$ drift on the oscillatory integrand — the residue result is exact where
   quadrature strains.
4. Interpret physically: the Lorentzian–exponential pair is the lineshape–lifetime relation
   behind [§6.24](../06-quantum-mechanics/time-dependent-perturbation.ipynb); a pole's imaginary
   part is a decay rate. (Prose part.)

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    [I_trig, I_fourier],
    [2 * np.pi / np.sqrt(3), np.pi * np.exp(-k7)],
    "the trigonometric and Fourier families close by residues (2π/√3 and (π/a)e^(−ka))",
    rtol=1e-4,
)
validate.close(
    [I_trig_direct, I_theta],
    [I_trig, I_trig],
    "two independent routes agree: raw unit-circle contour and direct θ-quadrature",
    rtol=1e-8,
)
validate.check(
    drift < 1e-2,
    "plain quad lands near the answer but strains on the oscillatory integrand; the residue is exact",
    f"quad drift {drift:.1e}",
)

## Exercise 8 — The keyhole: an integral around a branch cut

The finale, and a deliberate pre-echo: the Bose and Fermi integrals of
[§7.3](statistical-toolkit.ipynb) have exactly this
anatomy. The branch cut, bookkeeping until now, becomes the working part of the machine.
Cite {eq}`eq-branch-cuts`, {eq}`eq-real-integrals`.

1. For $\int_0^\infty x^{\alpha-1}/(1+x)\,dx$ with $\alpha = \tfrac13$, build the branch
   $z^{\alpha-1} = e^{(\alpha-1)\log z}$ with $\arg z \in [0, 2\pi)$ (fold `numpy.angle` into
   $[0, 2\pi)$ with `numpy.mod`), so the cut lies along the *positive* real axis, and assemble
   the keyhole contour's four legs numerically: two straight edges hugging the cut (transformed
   to log-space and integrated with `numpy.trapezoid`), a large circle of radius $R$, and a
   small circle of radius $\varepsilon$ (both with `contour_integral`).
2. Show numerically that the two straight edges differ by the phase factor $e^{2\pi i(\alpha-1)}$
   — they do *not* cancel; the cut is what makes the method work.
3. Confirm the four legs sum to $2\pi i$ times the residue at $z = -1$ (extracted with
   `residue_numeric` on the same branch), derive the closed form $\pi/\sin(\pi\alpha)$, and
   evaluate it from the *extracted* residue.
4. Confirm numerically against `scipy.integrate.quad`, and note in prose what changes when the
   denominator becomes $e^x - 1$ or $e^x + 1$ (the integrals of
   [§7.3](statistical-toolkit.ipynb) — the shape survives).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    edge_ratio,
    phase_expected,
    "the keyhole's edges differ by the branch phase −e^(2πiα): the cut prevents cancellation",
    rtol=1e-6,
)
validate.close(
    total_legs,
    rhs_theorem,
    "the four legs of the keyhole sum to 2πi times the extracted residue at z = −1",
    rtol=1e-5,
)
validate.close(
    I_keyhole,
    np.pi / np.sin(np.pi * alpha),
    "the keyhole contour gives ∫ x^(α−1)/(1+x) dx = π/sin(πα)",
    rtol=1e-6,
)
validate.close(
    I_keyhole,
    I_quad_key,
    "the residue-theorem value matches scipy.integrate.quad on the split, singular integral",
    rtol=1e-6,
)

## Exercise 9 — Rigidity as a tool

Everything in this notebook followed from one demand — that a derivative exist in the complex
sense — and the demand turned out to be nearly tyrannical. Functions that meet it are determined
inside by their boundary, are infinitely smooth without being asked, solve Laplace's equation
twice over, and surrender their closed-contour integrals to a handful of numbers sitting at their
singularities. We spent that rigidity on integrals: rational, trigonometric, oscillatory, and one
wrapped around a branch cut, each falling to the same three moves — close, locate, read off. And
the spending was not merely elegant: on the oscillatory Fourier integral the residue answer was
exact where a good adaptive quadrature warned and drifted, because a method that works with the
*analytic structure* of an integrand is playing a different game from one that samples its values.

It is worth savouring how strange this is. In real analysis, differentiability is cheap and buys
almost nothing — a function can be differentiable once and behave abominably thereafter. In
complex analysis it is expensive and buys everything. The physicist's luck is that nature keeps
writing analytic functions: the next notebook ([§7.2](complex-analysis-applications.ipynb))
points the machine at physics proper, where
the boundary values of response functions (Kramers–Kronig), the frequency sums of thermal physics
(Matsubara), and the asymptotics of the large-$N$ world (steepest descent, which finally *derives*
the Stirling formula that Volume V borrowed) are all consequences of analyticity — and causality
alone, we will see, is enough to force it. The notebook after that
([§7.3](statistical-toolkit.ipynb)) computes the integrals
this whole volume runs on, and their anatomy is already in hand: they are the keyhole of
Exercise 8 with a Boltzmann factor in the denominator.

## Notebook summary

Volume VII opens its mathematical arsenal with the core machinery of complex analysis, built
computationally and put to work immediately on the classic integral families.

- **Analyticity is the Cauchy–Riemann equations** {eq}`eq-cauchy-riemann`: equal limits from all
  directions force $u_x = v_y$, $u_y = -v_x$ — a sharp, checkable property (residuals $10^{-10}$
  for $z^2, e^z$; exactly $2$ for $\bar z$), and the real and imaginary parts of any analytic
  function are harmonic {eq}`eq-harmonic`, the electrostatics of Volume III in two dimensions.
- **Branch cuts are bookkeeping for multivaluedness** {eq}`eq-branch-cuts`: the principal
  logarithm (numpy's convention) jumps by $2\pi i$ across the negative axis, $\sqrt z$ returns
  from a loop as its own negative, and the cut can be *relocated* at will — later, deliberately,
  onto the positive axis.
- **Cauchy's theorem** {eq}`eq-cauchy-theorem`: $\oint f\,dz = 0$ over analytic territory
  (verified at $10^{-16}$), so contours deform freely; for the non-analytic $\bar z$ the integral
  instead measures the enclosed area exactly.
- **The integral formula** {eq}`eq-integral-formula`: boundary values reconstruct interior ones
  ($e^a$ to twelve digits), analytic functions are infinitely differentiable, and the
  reconstruction degrades on schedule ($e^{-Nd}$) as a pole grazes the discretized path.
- **Laurent series and residues** {eq}`eq-laurent`: only $c_{-1}$ survives closed-contour
  integration, so a small-circle contour extracts residues of any pole order — and, with extra
  powers inserted, the entire series ($c_n = 1/(n+2)!$ for $e^z/z^2$, read off numerically).
- **The residue theorem** {eq}`eq-residue-theorem` and **the four families**
  {eq}`eq-real-integrals`: $\pi/\sqrt2$ by a semicircle whose arc dies like $R^{-3}$ (measured);
  $2\pi/\sqrt3$ by walking the unit circle; $(\pi/a)e^{-|k|a}$ by Jordan's lemma — the
  Lorentzian–exponential lifetime pair of
  [§6.24](../06-quantum-mechanics/time-dependent-perturbation.ipynb), where the residue is exact
  while `quad` warns and
  drifts; and $\pi/\sin\pi\alpha$ by a keyhole around a branch cut whose edges refuse to cancel
  by precisely the factor $e^{2\pi i\alpha}$.

The rigidity of analytic functions is not a curiosity but a computational instrument — and the
keyhole that closed this notebook is, one Boltzmann factor away, the Bose and Fermi integrals on
which the rest of the volume runs.

## Outlook

- **The physicist's applications ([§7.2](complex-analysis-applications.ipynb)).** Principal
  values and the Sokhotski–Plemelj formula; the
  Kramers–Kronig relations, where causality alone forces the analyticity this notebook assumed;
  Matsubara frequency sums, which trade thermal sums for residue ladders; and steepest descent,
  which re-derives the Stirling formula of
  [§5.3](../05-classical-stat-mech/large-n-limit.ipynb) and grows into the saddle-point method of the
  large-$N$ world.
- **The statistical toolkit ([§7.3](statistical-toolkit.ipynb)).** Densities of states, the Gamma
  and zeta functions, the
  polylogarithms living on their branch cuts, the Bose and Fermi integrals (whose keyhole anatomy
  Exercise 8 rehearsed), and the Sommerfeld expansion.
- **Conformal mapping** — the harmonic-function connection of Exercise 1 grown into a method for
  two-dimensional electrostatics and ideal flow: a horizon, named.
- **Cross-reference** [§6.1](../06-quantum-mechanics/complex-vector-spaces.ipynb) (the algebra of
  $\mathbb{C}$), Volume III (harmonic potentials),
  [§5.3](../05-classical-stat-mech/large-n-limit.ipynb)
  (Stirling), [§6.24](../06-quantum-mechanics/time-dependent-perturbation.ipynb) (Lorentzian
  lineshapes and lifetimes), and forward to [§7.2](complex-analysis-applications.ipynb),
  [§7.3](statistical-toolkit.ipynb).

In [ ]:
from ecp.style import footer

footer()